# fastsae `hb_area()` engines on an NVIDIA GPU (Google Colab)

This notebook measures `hb_area()` with **INLA**, **JAX on the CPU**, **JAX on the CUDA GPU** and
**tipsae** (Stan, Beta model) on spatial (Besag ICAR) area-level models. The results feed the
pkgdown article
[hb_area() engines](https://ridsonap.github.io/fastsae/articles/hb-area-engines.html).

**Before running:** *Runtime → Change runtime type →* **R** and **T4 GPU**. Then *Runtime → Run all*.

Installation takes about 15–25 minutes. The benchmark then runs in the background for a few hours.
Keep this tab open. The results file is updated after every engine. If the runtime restarts, run
all cells again: finished parts are skipped.

In [1]:
# 1. Check the GPU
cat(system("nvidia-smi", intern = TRUE), sep = "\n")

Wed Oct 14 07:29:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. System libraries for sf / spdep (needed by tipsae)
invisible(system("apt-get -qq update && apt-get -qq install -y libudunits2-dev libgdal-dev libgeos-dev libproj-dev python3-venv", intern = TRUE))
cat("done\n")

done


In [3]:
# 3. R packages: binaries from Posit Package Manager, INLA from its own repository, fastsae from GitHub
codename <- system("lsb_release -cs", intern = TRUE)
options(
  repos = c(P3M = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", codename)),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
                          paste(getRversion(), R.version$platform, R.version$arch, R.version$os)),
  Ncpus = parallel::detectCores()
)
install.packages(c("remotes", "sf", "spdep", "tipsae", "reticulate", "ggplot2"))
install.packages("INLA", repos = c(getOption("repos"), INLA = "https://inla.r-inla-download.org/R/stable"),
                 dependencies = TRUE)
remotes::install_github("ridsonap/fastsae", upgrade = "never")
packageVersion("fastsae")

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘colorspace’, ‘fracdiff’, ‘lmtest’, ‘timeDate’, ‘urca’, ‘zoo’, ‘RcppArmadillo’, ‘Deriv’, ‘forecast’, ‘numDeriv’, ‘doBy’, ‘SparseM’, ‘MatrixModels’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘abind’, ‘tensorA’, ‘distributional’, ‘carData’, ‘Formula’, ‘pbkrtest’, ‘quantreg’, ‘lme4’, ‘checkmate’, ‘matrixStats’, ‘posterior’, ‘corrplot’, ‘car’, ‘wk’, ‘proxy’, ‘inline’, ‘gridExtra’, ‘loo’, ‘QuickJSR’, ‘ggrepel’, ‘ggsci’, ‘cowplot’, ‘ggsignif’, ‘polynom’, ‘rstatix’, ‘rbibutils’, ‘classInt’, ‘s2’, ‘units’, ‘spData’, ‘deldir’, ‘e1071’, ‘sp’, ‘rstan’, ‘ggpubr’, ‘Rdpack’, ‘BH’, ‘RcppEigen’, ‘RcppParallel’, ‘StanHeaders’, ‘RcppTOML’, ‘here’, ‘png’


Warning message in install.packages(c("remotes", "sf", "spdep", "tipsae", "reticulate", :
“installation of package ‘tipsae’ failed”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Warning message:
“dependencies 

── R CMD build ─────────────────────────────────────────────────────────────────
* checking for file ‘/tmp/Rtmp4wi1Tn/remotes419256da3db/ridsonap-fastsae-c7870da/DESCRIPTION’ ... OK
* preparing ‘fastsae’:
* checking DESCRIPTION meta-information ... OK
* cleaning src
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
* building ‘fastsae_1.0.0.tar.gz’



Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



[1] ‘1.0.0’

In [41]:
# 4. Python side: JAX with CUDA 12 and NumPyro, in the virtualenv "r-numpyro-gpu"
Sys.unsetenv("RETICULATE_PYTHON")  # Colab may point reticulate at its own Python

# Step 1: Create venv without pip to avoid the ensurepip error
system("bash -c 'python3 -m venv --without-pip /root/.virtualenvs/r-numpyro-gpu'")

# Step 2: Manually bootstrap pip inside the environment using get-pip.py
system("bash -c 'curl -sS https://bootstrap.pypa.io/get-pip.py | /root/.virtualenvs/r-numpyro-gpu/bin/python3'")

# Step 3: Install the required libraries
cmd <- "/root/.virtualenvs/r-numpyro-gpu/bin/pip install numpy scipy numpyro==0.19.0 jax[cuda12]==0.5.0"
cat("Running installation of libraries...\n")
log_out <- system(paste0("bash -c '", cmd, "' 2>&1"), intern = TRUE)
cat(log_out, sep = "\n")

# Force reticulate to use the created virtual environment
reticulate::use_virtualenv("r-numpyro-gpu", required = TRUE)

# Check in a fresh R process (JAX fixes its device per process): it must list a CUDA device
cat(system("Rscript -e 'fastsae::check_numpyro_available(device = \"cuda\")' 2>&1", intern = TRUE), sep = "\n")

Running installation of libraries...
✔ JAX devices found: cuda:0
[1] TRUE


In [38]:
# 5. Start the benchmark in the background (INLA, JAX CPU, JAX CUDA, tipsae)
download.file("https://raw.githubusercontent.com/ridsonap/fastsae/main/benchmarks/bench_hb_area_engines.R",
              "bench_hb_area_engines.R", quiet = TRUE)
out <- "/content/hb_engine_benchmark_colab_t4.rds"
# The benchmark processes inherit this: use the JAX/NumPyro virtualenv from step 4
Sys.setenv(RETICULATE_PYTHON = reticulate::virtualenv_python("r-numpyro-gpu"))
system2("Rscript", c("bench_hb_area_engines.R", "--engines=inla,cpu,cuda,tipsae", paste0("--out=", out)),
        stdout = "bench.log", stderr = "bench.log", wait = FALSE)

In [39]:
# 6. Progress: run this cell again to refresh. "exit" or "Saved" means finished.
log <- readLines("bench.log", warn = FALSE)
cat(grep(" n = |Error|Saved|failed", log, value = TRUE), sep = "\n")
cat("\nRunning:", any(grepl("bench_hb_area_engines", system("ps aux", intern = TRUE))), "\n")



Running: TRUE 


In [40]:
# 7. Results so far
res <- readRDS(out)
res$timings
res$agreement

machine,design,engine,n,time_first,time_warm,device,rhat_max
<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<chr>,<dbl>
"Google Colab, Tesla T4",gaussian,inla,100,0.899,0.482,cpu (INLA),NA
"Google Colab, Tesla T4",gaussian,inla,1000,1.220,1.090,cpu (INLA),NA
"Google Colab, Tesla T4",gaussian,inla,5000,12.695,13.889,cpu (INLA),NA


NULL

**8. Download the results:** open the *Files* panel (folder icon on the left), right-click
`hb_engine_benchmark_colab_t4.rds` → *Download*. Send this file to be added to the article (it is
saved as `inst/extdata/hb_engine_benchmark_colab_t4.rds` in the repository).

If the session is about to time out, download the file anyway. It holds every finished engine,
and a later run with the file in `/content` continues from there.